# Premier League Data Prep — 'What Wins the Premier League?'

Dit notebook laadt alle seizoensbestanden automatisch, voegt ze samen en exporteert 4 Tableau-ready CSVs:

| Output file | Gebruik in Tableau |
|---|---|
| `tableau_standings.csv` | Seizoensstand per team — tijdlijn, scatter MV vs punten |
| `tableau_match_records.csv` | Per wedstrijd per team — home/away analyse, team stats |
| `tableau_home_away.csv` | Home vs away winpct per seizoen — line chart |
| `tableau_player_season.csv` | Spelerstatistieken per seizoen — top spelers, ratings |

In [14]:
import pandas as pd
import numpy as np
import glob
import os

# Mappen naar je bestanden
MATCH_DIR = './matches'   # map met alle *_raw.csv bestanden
PLAYER_DIR = './players'  # map met alle *_players.csv bestanden

OUTPUT_DIR = './tableau_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Match map:', os.path.abspath(MATCH_DIR))
print('Player map:', os.path.abspath(PLAYER_DIR))

Match map: C:\Users\semwi\FPL-Core-Insights_Thesis\data\Seasonal data\matches
Player map: C:\Users\semwi\FPL-Core-Insights_Thesis\data\Seasonal data\players


## 1. Alle bestanden inladen

In [15]:
def fix_season(s):
    """Normaliseert seizoenformaten: '23/24' -> '2023-2024'"""
    s = str(s).strip()
    if '/' in s:
        a, b = s.split('/')
        return f'20{a}-20{b}'
    return s

def load_all_seasons(data_dir, pattern, file_type='match'):
    """Laadt alle CSV bestanden die matchen met het patroon en voegt ze samen."""
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    
    if not files:
        raise FileNotFoundError(f'Geen bestanden gevonden voor patroon: {pattern} in {data_dir}')
    
    print(f'Gevonden {len(files)} {file_type} bestanden:')
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, low_memory=False, on_bad_lines='skip')
            print(f'  {os.path.basename(f)}: {len(df)} rijen, {len(df.columns)} kolommen')
            dfs.append(df)
        except Exception as e:
            print(f'  SKIP {os.path.basename(f)}: {e}')
    
    combined = pd.concat(dfs, ignore_index=True)
    combined['season'] = combined['season'].apply(fix_season)
    print(f'Totaal: {len(combined)} rijen')
    print(f'Seizoenen: {sorted(combined["season"].unique())}')
    return combined

# Match files laden
print('=== MATCH FILES ===')
matches_raw = load_all_seasons(MATCH_DIR, '*_raw.csv', 'match')

print()
print('=== PLAYER FILES ===')
players_raw = load_all_seasons(PLAYER_DIR, '*_players.csv', 'player')

# Filter: alleen seizoenen vanaf 2020-2021
SEASONS = ['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']
matches_raw = matches_raw[matches_raw['season'].isin(SEASONS)].copy()
players_raw = players_raw[players_raw['season'].isin(SEASONS)].copy()

print(f'\nNa filter — matches: {len(matches_raw)}, players: {len(players_raw)}')
print(f'Seizoenen: {sorted(matches_raw["season"].unique())}')

=== MATCH FILES ===
Gevonden 10 match bestanden:
  2016-2017_raw.csv: 38 rijen, 80 kolommen
  2017-2018_raw.csv: 293 rijen, 84 kolommen
  2018-2019_raw.csv: 203 rijen, 82 kolommen
  2019-2020_raw.csv: 367 rijen, 84 kolommen
  2020-2021_raw.csv: 392 rijen, 98 kolommen
  2021-2022_raw.csv: 415 rijen, 96 kolommen
  2022-2023_raw.csv: 402 rijen, 92 kolommen
  2023-2024_raw.csv: 380 rijen, 106 kolommen
  2024-2025_raw.csv: 381 rijen, 108 kolommen
  2025-2026_raw.csv: 387 rijen, 112 kolommen
Totaal: 3258 rijen
Seizoenen: ['2016-2017', '2017-2018', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']

=== PLAYER FILES ===
Gevonden 12 player bestanden:
  2014-2015_players.csv: 13674 rijen, 67 kolommen
  2015-2016_players.csv: 22890 rijen, 67 kolommen
  2016-2017_players.csv: 26378 rijen, 67 kolommen
  2017-2018_players.csv: 17058 rijen, 67 kolommen
  2018-2019_players.csv: 13679 rijen, 67 kolommen
  2019-2020_players.csv: 14056 rijen, 67 kolommen
  2020-20

## 2. Match data opschonen

In [16]:
matches = matches_raw.copy()

# Timestamp van Unix naar datetime
matches['date'] = pd.to_datetime(matches['timestamp'], unit='s', errors='coerce')

# Goals naar int
matches = matches.dropna(subset=['home_goals', 'away_goals'])
matches['home_goals'] = matches['home_goals'].astype(float).astype(int)
matches['away_goals'] = matches['away_goals'].astype(float).astype(int)

# Numerieke teamstat kolommen
team_stat_cols = [
    'accurateCross', 'accurateLongBalls', 'accuratePasses',
    'aerialDuelsPercentage', 'ballPossession', 'ballRecovery',
    'bigChanceCreated', 'bigChanceMissed', 'bigChanceScored',
    'blockedScoringAttempt', 'cornerKicks', 'dispossessed',
    'dribblesPercentage', 'duelWonPercent', 'finalThirdEntries',
    'fouls', 'goalkeeperSaves', 'hitWoodwork', 'interceptionWon',
    'passes', 'shotsOffGoal', 'shotsOnGoal', 'totalClearance',
    'totalShotsInsideBox', 'totalShotsOnGoal', 'totalShotsOutsideBox',
    'totalTackle', 'wonTacklePercent'
]

for col in team_stat_cols:
    for side in ['home', 'away']:
        full_col = f'{side}_{col}'
        if full_col in matches.columns:
            matches[full_col] = pd.to_numeric(matches[full_col], errors='coerce')

# Resultaat kolom
def get_result(row):
    if row['home_goals'] > row['away_goals']: return 'Home Win'
    elif row['home_goals'] < row['away_goals']: return 'Away Win'
    else: return 'Draw'

matches['result'] = matches.apply(get_result, axis=1)

print(f'Wedstrijden na cleaning: {len(matches)}')
print(f'Resultaten: {matches["result"].value_counts().to_dict()}')
matches[['season', 'home_team', 'away_team', 'home_goals', 'away_goals', 'result', 'date']].head(3)

Wedstrijden na cleaning: 2263
Resultaten: {'Home Win': 974, 'Away Win': 756, 'Draw': 533}


,season,home_team,away_team,home_goals,away_goals,result,date
901,2020-2021,Aston Villa,Everton,0,0,Draw,2021-05-13 17:00:00
902,2020-2021,Manchester United,Liverpool,2,4,Away Win,2021-05-13 19:15:00
903,2020-2021,Newcastle United,Manchester City,3,4,Away Win,2021-05-14 19:00:00


## 3. Match records unpivoten (home + away als aparte rijen)

In [17]:
def build_match_records(df):
    """Zet wide match data om naar long format: elke wedstrijd = 2 rijen (home + away)"""
    records = []

    for _, row in df.iterrows():
        hg, ag = row['home_goals'], row['away_goals']

        if hg > ag:
            h_pts, a_pts = 3, 0
            h_res, a_res = 'Win', 'Loss'
        elif hg < ag:
            h_pts, a_pts = 0, 3
            h_res, a_res = 'Loss', 'Win'
        else:
            h_pts, a_pts = 1, 1
            h_res, a_res = 'Draw', 'Draw'

        base = {
            'match_id': row['match_id'],
            'season': row['season'],
            'round': row.get('round'),
            'date': row.get('date'),
            'venue_name': row.get('venue'),
            'referee': row.get('referee'),
            'attendance': pd.to_numeric(row.get('attendance'), errors='coerce'),
        }

        for side, team, opp, gf, ga, pts, res in [
            ('Home', row['home_team'], row['away_team'], hg, ag, h_pts, h_res),
            ('Away', row['away_team'], row['home_team'], ag, hg, a_pts, a_res)
        ]:
            prefix = side.lower()
            record = {**base,
                'team': team,
                'opponent': opp,
                'venue': side,
                'goals_for': gf,
                'goals_against': ga,
                'goal_difference': gf - ga,
                'points': pts,
                'result': res,
                'win': 1 if res == 'Win' else 0,
                'draw': 1 if res == 'Draw' else 0,
                'loss': 1 if res == 'Loss' else 0,
            }
            # Voeg teamstatistieken toe
            for col in team_stat_cols:
                full_col = f'{prefix}_{col}'
                if full_col in df.columns:
                    record[col] = row.get(full_col)

            records.append(record)

    return pd.DataFrame(records)


match_records = build_match_records(matches)
print(f'Match records: {len(match_records)} rijen')
match_records.head(3)

Match records: 4526 rijen


,match_id,season,round,date,venue_name,referee,attendance,team,opponent,venue,...,interceptionWon,passes,shotsOffGoal,shotsOnGoal,totalClearance,totalShotsInsideBox,totalShotsOnGoal,totalShotsOutsideBox,totalTackle,wonTacklePercent
0,9501660,2020-2021,19.0,2021-05-13 17:00:00,Villa Park,Martin Atkinson,0.0,Aston Villa,Everton,Home,...,6.0,502.0,7.0,2.0,16.0,7.0,13.0,6.0,7.0,5.0
1,9501660,2020-2021,19.0,2021-05-13 17:00:00,Villa Park,Martin Atkinson,0.0,Everton,Aston Villa,Away,...,8.0,362.0,9.0,5.0,17.0,10.0,15.0,5.0,9.0,4.0
2,9507199,2020-2021,34.0,2021-05-13 19:15:00,Old Trafford,Anthony Taylor,0.0,Manchester United,Liverpool,Home,...,9.0,511.0,8.0,3.0,12.0,13.0,18.0,5.0,22.0,13.0


## 4. Seizoensstand berekenen

In [18]:
agg_dict = {
    'match_id': 'count',
    'points': 'sum',
    'win': 'sum',
    'draw': 'sum',
    'loss': 'sum',
    'goals_for': 'sum',
    'goals_against': 'sum',
    'attendance': 'mean',
}

# Voeg teamstat gemiddelden toe
for col in team_stat_cols:
    if col in match_records.columns:
        agg_dict[col] = 'mean'

standings = match_records.groupby(['season', 'team']).agg(agg_dict).reset_index()
standings.rename(columns={'match_id': 'matches'}, inplace=True)

standings['goal_difference'] = standings['goals_for'] - standings['goals_against']
standings['win_pct'] = standings['win'] / standings['matches'] * 100
standings['goals_per_game'] = standings['goals_for'] / standings['matches']

# Rangschikking per seizoen (op punten, dan doelsaldo)
standings['position'] = standings.groupby('season').apply(
    lambda x: x[['points', 'goal_difference', 'goals_for']]
    .apply(tuple, axis=1).rank(ascending=False, method='min')
).reset_index(level=0, drop=True).astype(int)

standings['is_champion'] = standings['position'] == 1
standings['top_4'] = standings['position'] <= 4
standings['relegated'] = standings['position'] >= 18

print('=== KAMPIOENEN PER SEIZOEN ===')
champs = standings[standings['is_champion']][['season','team','points','goal_difference','win_pct']]
print(champs.sort_values('season').to_string(index=False))

=== KAMPIOENEN PER SEIZOEN ===
   season            team  points  goal_difference   win_pct
2020-2021 Manchester City      86               51 71.052632
2021-2022 Manchester City      93               73 76.315789
2022-2023 Manchester City      89               61 73.684211
2023-2024 Manchester City      91               62 73.684211
2024-2025       Liverpool      84               45 65.789474
2025-2026         Arsenal      82               45 67.567568


C:\Users\semwi\AppData\Local\Temp\ipykernel_14852\1423929209.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  standings['position'] = standings.groupby('season').apply(


## 5. Home/Away winpercentage per seizoen

In [19]:
home_away = (
    match_records
    .groupby(['season', 'venue', 'result'])
    .size()
    .reset_index(name='count')
)

totals = (
    match_records
    .groupby(['season', 'venue'])
    .size()
    .reset_index(name='total')
)

home_away = home_away.merge(totals, on=['season', 'venue'])
home_away['percentage'] = home_away['count'] / home_away['total'] * 100

# Ook overall per seizoen voor Tableau
overall_results = (
    matches
    .groupby(['season', 'result'])
    .size()
    .reset_index(name='count')
)
season_totals = matches.groupby('season').size().reset_index(name='total')
overall_results = overall_results.merge(season_totals, on='season')
overall_results['percentage'] = overall_results['count'] / overall_results['total'] * 100

print('Home/Away sample:')
print(home_away.head(6).to_string(index=False))

Home/Away sample:
   season venue result  count  total  percentage
2020-2021  Away   Draw     83    380   21.842105
2020-2021  Away   Loss    144    380   37.894737
2020-2021  Away    Win    153    380   40.263158
2020-2021  Home   Draw     83    380   21.842105
2020-2021  Home   Loss    153    380   40.263158
2020-2021  Home    Win    144    380   37.894737


## 6. Player data opschonen

In [20]:
players = players_raw.copy()

# Timestamp naar datum
players['date'] = pd.to_datetime(players['timestamp'], unit='s', errors='coerce')

# Team kolom
players['team'] = np.where(players['side'] == 'home', players['home_team'], players['away_team'])

# Numerieke kolommen
num_cols = [
    'rating', 'goals', 'goal_assist', 'minutes_played', 'market_value',
    'total_pass', 'accurate_pass', 'total_shots', 'on_target',
    'duel_won', 'duel_lost', 'total_tackle', 'won_tackle',
    'interception_won', 'total_clearance', 'key_pass',
    'big_chance_created', 'big_chance_missed', 'was_fouled', 'fouls',
    'ball_recovery', 'aerial_won', 'aerial_lost', 'saves',
    'acc_own_half_pass', 'acc_opp_half_pass', 'height'
]

for col in num_cols:
    if col in players.columns:
        players[col] = pd.to_numeric(players[col], errors='coerce')

print(f'Player rows: {len(players)}')
print(f'Marktwaarde range: €{players["market_value"].min():,.0f} — €{players["market_value"].max():,.0f}')
print(f'Seizoenen: {sorted(players["season"].unique())}')

Player rows: 90467
Marktwaarde range: €23,000 — €218,000,000
Seizoenen: ['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']


## 7. Spelerstatistieken aggregeren per seizoen

In [21]:
player_season = (
    players
    .groupby(['season', 'player_id', 'player_name', 'team', 'position', 'nationality'])
    .agg(
        appearances=('match_id', 'nunique'),
        avg_rating=('rating', 'mean'),
        total_goals=('goals', 'sum'),
        total_assists=('goal_assist', 'sum'),
        total_minutes=('minutes_played', 'sum'),
        market_value=('market_value', 'mean'),  # echte euros, niet genormaliseerd
        total_passes=('total_pass', 'sum'),
        accurate_passes=('accurate_pass', 'sum'),
        total_shots=('total_shots', 'sum'),
        shots_on_target=('on_target', 'sum'),
        duels_won=('duel_won', 'sum'),
        duels_lost=('duel_lost', 'sum'),
        tackles=('total_tackle', 'sum'),
        won_tackles=('won_tackle', 'sum'),
        interceptions=('interception_won', 'sum'),
        key_passes=('key_pass', 'sum'),
        big_chances_created=('big_chance_created', 'sum'),
        saves=('saves', 'sum'),
        avg_height=('height', 'mean'),
    )
    .reset_index()
)

# Filter: minimaal 5 wedstrijden gespeeld
player_season = player_season[player_season['appearances'] >= 5].copy()

# Berekende velden
player_season['goal_contributions'] = player_season['total_goals'] + player_season['total_assists']
player_season['goals_per_90'] = np.where(
    player_season['total_minutes'] > 0,
    player_season['total_goals'] / player_season['total_minutes'] * 90, np.nan
)
player_season['assists_per_90'] = np.where(
    player_season['total_minutes'] > 0,
    player_season['total_assists'] / player_season['total_minutes'] * 90, np.nan
)
player_season['pass_accuracy_pct'] = np.where(
    player_season['total_passes'] > 0,
    player_season['accurate_passes'] / player_season['total_passes'] * 100, np.nan
)
player_season['shot_accuracy_pct'] = np.where(
    player_season['total_shots'] > 0,
    player_season['shots_on_target'] / player_season['total_shots'] * 100, np.nan
)
player_season['tackle_win_pct'] = np.where(
    player_season['tackles'] > 0,
    player_season['won_tackles'] / player_season['tackles'] * 100, np.nan
)
player_season['duel_win_pct'] = np.where(
    (player_season['duels_won'] + player_season['duels_lost']) > 0,
    player_season['duels_won'] / (player_season['duels_won'] + player_season['duels_lost']) * 100, np.nan
)

# Koppel seizoensstand
player_season = player_season.merge(
    standings[['season', 'team', 'position', 'points', 'is_champion', 'top_4', 'relegated']],
    on=['season', 'team'], how='left'
)

print(f'Speler-seizoen records: {len(player_season)}')
print(f'Unieke spelers: {player_season["player_id"].nunique()}')

# Top 10 op gemiddelde rating
top10 = (
    player_season[player_season['appearances'] >= 15]
    .groupby('player_name')[['avg_rating','total_goals','total_assists','market_value']]
    .mean()
    .sort_values('avg_rating', ascending=False)
    .head(10)
)
print('\nTop 10 spelers (gem. rating, min 15 wedstrijden/seizoen):')
print(top10.round(2).to_string())

Speler-seizoen records: 3943
Unieke spelers: 1452

Top 10 spelers (gem. rating, min 15 wedstrijden/seizoen):
                        avg_rating  total_goals  total_assists  market_value
player_name                                                                 
James Rodríguez               7.73         5.00           4.00     1900000.0
Kevin De Bruyne               7.72         6.60          10.20    15500000.0
Bruno Fernandes               7.56         8.86           8.86    37000000.0
Arijanet Murić                7.54         0.00           0.00     6900000.0
Harry Kane                    7.49        23.33           8.67    71000000.0
Rodri                         7.49         4.00           3.80    67775000.0
Asmir Begović                 7.45         0.00           0.00      210000.0
Thomas Strakosha              7.40         0.00           0.00     2300000.0
Kristoffer Klaesson           7.40         0.00           0.00     1000000.0
Trent Alexander-Arnold        7.36         2

## 8. Exporteren naar Tableau

In [22]:
exports = {
    'tableau_standings.csv': standings,
    'tableau_match_records.csv': match_records,
    'tableau_home_away.csv': home_away,
    'tableau_player_season.csv': player_season,
}

for filename, df in exports.items():
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'✓ {filename} — {len(df)} rijen, {len(df.columns)} kolommen')

print(f'\nAlle bestanden staan in: {os.path.abspath(OUTPUT_DIR)}')
print('Laad alle 4 CSVs in Tableau en join op season + team.')

✓ tableau_standings.csv — 121 rijen, 45 kolommen
✓ tableau_match_records.csv — 4526 rijen, 46 kolommen
✓ tableau_home_away.csv — 36 rijen, 6 kolommen
✓ tableau_player_season.csv — 3943 rijen, 37 kolommen

Alle bestanden staan in: C:\Users\semwi\FPL-Core-Insights_Thesis\data\Seasonal data\tableau_output
Laad alle 4 CSVs in Tableau en join op season + team.


## 9. Sanity checks

In [23]:
print('=== KAMPIOENEN ===')
print(standings[standings['is_champion']][
    ['season','team','points','goal_difference','win_pct','ballPossession','totalShotsOnGoal']
].sort_values('season').round(1).to_string(index=False))

print('\n=== HOME/AWAY WINS (alle seizoenen) ===')
summary = match_records.groupby(['venue','result'])['match_id'].count().unstack(fill_value=0)
print(summary)

print('\n=== MARKTWAARDE RANGE PER SEIZOEN (kampioenen) ===')
champ_players = player_season[player_season['is_champion'] == True]
mv_range = champ_players.groupby('season')['market_value'].agg(['mean','max'])
mv_range.columns = ['gem_mv', 'max_mv']
print(mv_range.applymap(lambda x: f'€{x:,.0f}' if pd.notna(x) else 'n/a'))

=== KAMPIOENEN ===
   season            team  points  goal_difference  win_pct  ballPossession  totalShotsOnGoal
2020-2021 Manchester City      86               51     71.1            63.7              15.8
2021-2022 Manchester City      93               73     76.3            68.2              18.8
2022-2023 Manchester City      89               61     73.7            65.1              15.8
2023-2024 Manchester City      91               62     73.7            65.4              18.2
2024-2025       Liverpool      84               45     65.8            57.9              17.1
2025-2026         Arsenal      82               45     67.6            55.5              15.7

=== HOME/AWAY WINS (alle seizoenen) ===
result  Draw  Loss  Win
venue                  
Away     533   974  756
Home     533   756  974

=== MARKTWAARDE RANGE PER SEIZOEN (kampioenen) ===
                gem_mv        max_mv
season                              
2020-2021  €26,266,964  €110,000,000
2021-2022  €26,237,241 

C:\Users\semwi\AppData\Local\Temp\ipykernel_14852\3360113308.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(mv_range.applymap(lambda x: f'€{x:,.0f}' if pd.notna(x) else 'n/a'))


In [2]:
import pandas as pd

# Laad het originele bestand
df = pd.read_csv('tableau_output/tableau_match_records.csv')

# De Big Six teams
big_six = [
    'Arsenal',
    'Chelsea',
    'Liverpool',
    'Manchester City',
    'Manchester United',
    'Tottenham'
]

# Filter: behoud alleen wedstrijden waar minstens één van de Big Six speelt
# (elke wedstrijd staat 2x in het bestand - één rij per team)
mask = df['team'].isin(big_six) | df['opponent'].isin(big_six)
df_big_six = df[mask].copy()

print(f"Origineel aantal rijen: {len(df)}")
print(f"Na filter Big Six: {len(df_big_six)}")
print(f"\nTeams in gefilterde dataset:\n{sorted(df_big_six['team'].unique())}")

# Sla op als nieuw CSV bestand
df_big_six.to_csv('tableau_match_records_big_six.csv', index=False)
print("\nBestand opgeslagen als: tableau_match_records_big_six.csv")

Origineel aantal rijen: 4526
Na filter Big Six: 2026

Teams in gefilterde dataset:
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton & Hove Albion', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich Town', 'Leeds United', 'Leicester City', 'Liverpool', 'Liverpool FC', 'Luton Town', 'Manchester City', 'Manchester United', 'Newcastle United', 'Norwich City', 'Nottingham Forest', 'Sheffield United', 'Southampton', 'Sunderland', 'Tottenham Hotspur', 'Watford', 'West Bromwich Albion', 'West Ham United', 'Wolverhampton']

Bestand opgeslagen als: tableau_match_records_big_six.csv
